<a href="https://colab.research.google.com/github/niranjanappaji/LLM_Engineering/blob/main/3_LLM_Engg_Company_Brochure_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


##### **Company brochure generator using web scraping and an OpenAI model.**

**The workflow:**
1. Fetch a company's landing page and extract readable text and links.
2. Use an LLM to identify links that are useful for understanding the company.
3. Fetch the selected pages and combine their content.
4. Ask an LLM to turn the collected information into a concise Markdown brochure.
5. Optionally stream the generated brochure for a better interactive experience.


In [ ]:
# ==============================================================================
# [Code] Setup Environment & Dependencies
# ==============================================================================
# Install runtime dependencies when this file is executed in Google Colab/Jupyter.

!pip install -q requests bs4 selenium openai

import os
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from IPython.display import display, Markdown, update_display
import re
import json

print("Set Up Complete!")

Set Up Complete!


In [ ]:
# ==============================================================================
# [Code] Initialize and Constants
# ==============================================================================

from openai import OpenAI
from google.colab import userdata

def initialize_openai_api_client():

  try:
    api_key = userdata.get('OPENAI_API_KEY')
  except Exception:
    api_key = os.environ.get('OPENAI_API_KEY')

  if not api_key:
    print("Please set api_key either in os environments or colab secrets")

  os.environ['OPENAI_API_KEY'] = api_key.strip()
  return OpenAI()

openai = initialize_openai_api_client()
MODEL = 'gpt-5-nano'



In [ ]:
# ==============================================================================
# [Code] Web Scraper Implementation
# ==============================================================================

def fetch_website_contents(url):
  try:
    response = requests.get(
        url,
        headers={"User-Agent":"Mozilla/5.0"},
        timeout=10
    )

    soup = BeautifulSoup(response.text, "html.parser")

    links = [
        link.get("href") for link in soup.find_all("a", href=True)
    ]

    links = [link for link in links if link]

    for element in soup(["script", "style", "img", "input", "nav", "footer", "header"]):
      element.decompose()

    text = soup.get_text(" ", strip=True)
    cleaned = re.sub(r"\s+", ' ', text)
    cleaned = re.sub(r"http\S+", ' ', cleaned).strip()

    if len(cleaned) < 100:
      return None

    return {
        "text" : cleaned,
        "links" : links
    }

  except Exception as e:
    print(f"BS4 Error: {e}")
    return None



In [ ]:
# result = fetch_website_contents("https://us.adukale.com/")
# result["links"]

fetch_website_contents("https://us.adukale.com/")["links"]

['#main',
 'mailto:contact@homebitesusa.com',
 'https://www.facebook.com/AdukaleFoods',
 'https://www.instagram.com/adukalefoods',
 '/',
 '/',
 'https://us.adukale.com/customer_authentication/redirect?locale=en&region_country=US',
 '/cart',
 '/collections/snacks',
 '/collections/top-picks',
 '/collections/kodubales',
 '/collections/murukus',
 '/collections/mixtures',
 '/collections/masala-nuts',
 '/collections/chakkulis',
 '/collections/instants',
 '/collections/dosa-varieties',
 '/collections/idly-varieties',
 '/collections/poha-varieties',
 '/collections/upma-varieties',
 '/collections/masala-powders',
 '/collections/masala-powders',
 '/collections/chutney-powders',
 '/collections/desserts',
 '/collections/snacks',
 '/collections/top-picks',
 '/collections/kodubales',
 '/collections/murukus',
 '/collections/mixtures',
 '/collections/masala-nuts',
 '/collections/chakkulis',
 '/collections/instants',
 '/collections/dosa-varieties',
 '/collections/idly-varieties',
 '/collections/poha-va

In [ ]:
# ==============================================================================
# [Code] Build System Prompts
# ==============================================================================

link_system_prompts = """
You are provided with a list of links found on a website.
You are able to decide which of the links would be most relevant to include in a brochure about a company,
such as links to an About page, or a company page, or a products page or careers page.
You should respond in JSON as in the below example:

{
  "links":[
    {"type":"about page", "url":"https://example.com/about"},
    {"type":"products page", "url":"https://example.com/products"},
    {"type":"careers page", "url":"https://example.com/careers"},
  ]
}
"""

In [ ]:
# ==============================================================================
# [Code] Build User Prompts
# ==============================================================================

def get_user_prompts(url):

  user_prompt = f"""
  Here is the list of links found on a website {url} -
  Please decide which of the links would be most relevant to include in a brochure about a company,
  respond with full https url in JSON format.
  Do not include Terms of Service, Privacy, email and mailto links

  Links (some of these might be relevant):

  """

  links = fetch_website_contents(url)["links"]
  user_prompt += "\n".join(links)
  return user_prompt

In [ ]:
# ==============================================================================
# [Code] Select Relevant links
# ==============================================================================

def select_relevant_links(url):
  print(f"Selecting relevant links for {url} by model {MODEL}")
  response = openai.chat.completions.create(
      model=MODEL,
      messages=[
          {"role":"system", "content":link_system_prompts},
          {"role":"user", "content":get_user_prompts(url)}
      ],
          response_format={"type":"json_object"}
  )

  result = response.choices[0].message.content
  links = json.loads(result)
  print(f"Found {len(links['links'])} relevant links")
  return links



In [ ]:
select_relevant_links("https://us.adukale.com/")

Selecting relevant links for https://us.adukale.com/ by model gpt-5-nano
Found 17 relevant links


{'links': [{'type': 'about page', 'url': 'https://us.adukale.com/pages/about'},
  {'type': 'contact page', 'url': 'https://us.adukale.com/pages/contact'},
  {'type': 'collection page', 'url': 'https://us.adukale.com/collections/all'},
  {'type': 'collection page',
   'url': 'https://us.adukale.com/collections/best-sellers'},
  {'type': 'collection page',
   'url': 'https://us.adukale.com/collections/snacks'},
  {'type': 'collection page',
   'url': 'https://us.adukale.com/collections/masala-powders'},
  {'type': 'collection page',
   'url': 'https://us.adukale.com/collections/chutney-powders'},
  {'type': 'collection page',
   'url': 'https://us.adukale.com/collections/dosa-varieties'},
  {'type': 'collection page',
   'url': 'https://us.adukale.com/collections/idly-varieties'},
  {'type': 'collection page',
   'url': 'https://us.adukale.com/collections/upma-varieties'},
  {'type': 'collection page',
   'url': 'https://us.adukale.com/collections/poha-varieties'},
  {'type': 'collection

In [ ]:
# ==============================================================================
# [Code] Create Company Brochure
# ==============================================================================

def fetch_text_and_relevant_links(url):
  print(f"Fetching the website content and relevant links for {url} by model {MODEL}")
  content_result = fetch_website_contents(url)
  if content_result is None:
    return None

  contents = content_result["text"]

  rel_links = select_relevant_links(url)
  result = f"## Landing Page:\n\n {contents}\n ## Relevant links: \n"
  for link in rel_links['links']:
    link_result = fetch_website_contents(link['url'])
    if link_result is None:
      continue
    result += f"\n\n ### Link: {link['type']}\n"
    result += link_result["text"]
  return result


In [ ]:
print(fetch_text_and_relevant_links("https://us.adukale.com/"))

Fetching the website content and relevant links for https://us.adukale.com/ by model gpt-5-nano
Selecting relevant links for https://us.adukale.com/ by model gpt-5-nano
Found 7 relevant links
## Landing Page:

 Adukale - The Taste of Sankethi Cuisine - Snacks, Instants, and More Skip to content Email Facebook Instagram Previous Please note: Due to high volume, all orders placed today will be processed in 4-5 business days. Trusted by 60,000+ Happy Customers! 🌟 MOQ $30 | Free Shipping Over $100 Next Adukale Submit Account Open cart Shopping Cart Total: $0.00 USD 0 products in your cart $0.00 USD ( 0 ) Open menu Open cart 0 Submit Menu Close sidebar Email Facebook Instagram Shop by Category Snacks Masalas & Powders Chutney Powders Dosa Varieties Upma Varieties Our Bestsellers View all products Regular price $3.99 USD Kayi Kodubale | 180g Buy now Regular price $3.99 USD Nippattu | 180g Buy now Regular price $3.49 USD Instant Gojjavalakki (Tangy Tamarind Poha) | 250g Buy now Regular price 

In [ ]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages of a website
and creates a short brochure about a company for prospective customers, marketing agencies, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers, testimonials, and careers/jobs if you have the information.
"""

# https://pineconeindia.in/

In [ ]:
def get_brochure_user_prompt(company_name, url):
  user_prompt = f"""
  You are looking at a company called {company_name}
  Here are the contents of its landing page and other relevant pages,
  use this information to build a short brochure of the company in markdown without code blocks.
  \n\n
  """

  user_prompt += fetch_text_and_relevant_links(url)
  user_prompt = user_prompt[:5000] # Truncate if more than 5000 characters
  return user_prompt

In [ ]:
print(get_brochure_user_prompt("Pine Cone Home Decor & Handicrafts India", "https://pineconeindia.in/"))

Fetching the website content and relevant links for https://pineconeindia.in/ by model gpt-5-nano
Selecting relevant links for https://pineconeindia.in/ by model gpt-5-nano
Found 15 relevant links

  You are looking at a company called Pine Cone Home Decor & Handicrafts India
  Here are the contents of its landing page and other relevant pages,
  use this information to build a short brochure of the company in markdown without code blocks.
  


  ## Landing Page:

 Handmade Home Decor Items Online in India - PineConeIndia – Pine cone™ Home About us Expand submenu About us Collapse submenu About us Our Story Founder By Decor Expand submenu By Decor Collapse submenu By Decor Tray baskets Planters Laundry/ Storage Baskets Wall Decor Table Decor and Dining Macrame` By Craft Expand submenu By Craft Collapse submenu By Craft Assam Himachal Pradesh Jammu & Kashmir - Homeland Manipur Bags and Accessories Handcrafted Gifts for everyday and special day gifting Kids Decor - Pine Cone™ Mini Made t

In [ ]:
def create_brochure(company_name, url):
  response = openai.chat.completions.create(
      model=MODEL,
      messages=[
          {"role":"system", "content":brochure_system_prompt},
          {"role":"user", "content":get_brochure_user_prompt(company_name, url)}
      ]
  )

  result = response.choices[0].message.content
  display(Markdown(result))

In [ ]:
print(create_brochure("Pine Cone Home Decor & Handicrafts India", "https://pineconeindia.in/"))

Fetching the website content and relevant links for https://pineconeindia.in/ by model gpt-5-nano
Selecting relevant links for https://pineconeindia.in/ by model gpt-5-nano
Found 24 relevant links


# Pine Cone Home Decor & Handicrafts India

A craft-based home decor brand that celebrates India’s lesser-known artisans. Handmade, ethically produced, and designed to fit contemporary living with a Scandinavian-Boho aesthetic.

## Who we are
- Pine Cone™ is a craft-based home decor initiative that brings lesser known Indian crafts directly from artisans in remote rural areas to you.
- 90% of our artisans are women. We partner with them to upgrade skills, diversify their work, and transform traditional crafts into beautiful, utilitarian home decor.
- We’re proudly handmade and crafted in India, working with crafts from the North East, Himachal Pradesh, and Jammu regions.

## What we offer
- Handmade Home Decor Items, with a contemporary utilitarian vibe
- Categories:
  - Table Decor and Dining
  - Wall Decor
  - Macrame (Scandi-Boho)
  - Bags and Personal Accessories
  - Picnic baskets, trays, and bottle holders for outdoor meals
  - By Decor and By Craft collections
  - Made to Order options
- International shipping available
- Special note: Indian Rattan wine holder is a featured example of our cane craftsmanship

## Our impact and culture
- We celebrate craft traditions by giving them a contemporary feel to appeal to both minimalists and maximalists through design intervention.
- Direct-from-source approach supports artisans and sustains regional crafts.
- Fully handmade in India, with a focus on elevating traditional skills to modern, utilitarian use.

## Global reach and accessibility
- We ship internationally; for destinations outside India, please email contactus.pinecone@gmail.com or support@pineconeindia.in with your requirements and we’ll respond promptly.
- For outside-India inquiries, we’re ready to collaborate on your destination needs.

## Media and credibility
- Featured in notable outlets such as Times of India (Times Plus, Delhi-NCR), The Indian Express (Indulge), and Her Story, among others.
- Our Handcrafted Stories explore the ethos of our work and the people behind the pieces, offering insight into why our products matter.

## For customers, marketing partners, investors
- Customers: High-quality, handmade pieces that bring authentic Indian crafts into modern homes.
- Marketing agencies and retailers: We offer unique, ethically sourced products with a compelling storytelling angle around women artisans and regional crafts. (Please reach out via the contact emails above to discuss collaborations.)
- Investors: A scalable model rooted in fair-wage artisan partnerships, diverse regional crafts, and a growing international shipping footprint.

## Careers and opportunities
- Our site highlights our mission and product ranges but does not list specific career openings. If you’re interested in opportunities with Pine Cone, please contact us via the emails above to inquire about roles, internships, or partnerships.

## Get in touch
- For orders outside India and custom requirements: contactus.pinecone@gmail.com or support@pineconeindia.in
- General inquiries and collaborations: use the same contact channels

If you’d like, I can tailor this brochure to a specific audience (customers, agencies, investors, or recruits) or create a shorter one-page version.

None


In [ ]:
def stream_brochure(company_name, url):
  stream = openai.chat.completions.create(
      model = MODEL,
      messages = [
          {"role":"system", "content":brochure_system_prompt},
          {"role":"user", "content":get_brochure_user_prompt(company_name, url)}
      ],
      stream=True,
  )
  response = ""

  display_response = display(Markdown(""), display_id=True)
  for chunk in stream:
    response += chunk.choices[0].delta.content or ''
    update_display(Markdown(response), display_id = display_response.display_id)



In [ ]:
stream_brochure("Pine Cone Home Decor & Handicrafts India", "https://pineconeindia.in/")

Fetching the website content and relevant links for https://pineconeindia.in/ by model gpt-5-nano
Selecting relevant links for https://pineconeindia.in/ by model gpt-5-nano
Found 21 relevant links


Pine Cone Home Decor & Handicrafts India
A craft-based home decor brand sharing lesser known Indian crafts, handmade with love, directly from the source.

About Pine Cone
- We celebrate India’s lesser known crafts, handmade in remote parts of rural India.
- Our artisans are primarily women (about 90% of them).
- We upgrade craft skills to make art more utilitarian and commercial, transforming traditional crafts into beautiful home décor pieces.
- All products are completely handmade in India, with a contemporary, Scandinavian-Boho aesthetic.
- Our origins span multiple regions, including the North East, Himachal Pradesh, and Jammu & Kashmir.

Our Craft & Culture
- Craft-based home décor initiative focused on empowering artisans and sustaining traditional skills.
- Direct-from-source model that connects customers with the makers.
- Emphasis on multi-use, functional beauty in everyday living spaces.
- A culture of care, quality, and respectful collaboration with artisans in rural communities.

Products & Collections
- Handmade Home Decor Items
- Handcrafted Table Decor & Dining
- Wall Decor
- Macramé – Scandi-Boho
- Handmade Bags & Personal Accessories
- Made to Order
- Picnic essentials: Handcrafted picnic baskets, trays, and bottle holders for outdoor meals
- Featured item: Indian Rattan wine Holder (handcrafted cane, ring-by-ring created by artisans)

Our Makers
- Artisans from diverse Indian regions, including the North East, Himachal Pradesh, and Jammu & Kashmir.
- Emphasis on empowering women through skill upgrades and sustainable livelihoods.
- Each piece is handmade, unique, and carries the story of its maker.

International Shipping & How to Shop
- We ship outside India now. For international requirements, contact: 
  - contactus.pinecone@gmail.com
  - support@pineconeindia.in
- For any destination outside India, share your requirement and we will respond with next steps.

Media, Stories & What Our Customers Say
- Featured and covered in major outlets such as Times of India (Times Plus – Delhi NCR) and Indian Express (Indulge), among others.
- Handcrafted Stories highlight the human side of our crafts, with posts like:
  - Why Some Things Never Leave the House
  - The Things We Shared
  - Functional Beauty: Why home decor should be multi-utility and versatile
- Customers buy for everyday living and gifting—our collections are built for personal use and thoughtful presents.

Newsletter & Updates
- Stay in the loop with new arrivals, promotions, and stories by subscribing to our newsletter.

Careers & Join Us
- No specific career listings are shown on the site. If you’re interested in opportunities, please reach out via the contact emails above to discuss potential roles or partnerships.

Get in Touch
- Email for orders and international requirements: contactus.pinecone@gmail.com
- Email for general support: support@pineconeindia.in
- Newsletter signup available on the site to receive promotions and new product announcements.

Why Choose Pine Cone
- Direct-from-source, ethically sourced Indian crafts
- Superior, handmade pieces with contemporary design
- Strong focus on women artisans and rural empowerment
- International shipping options to bring Indian craftsmanship to homes worldwide

Call to action
- Explore our collections online, discover the stories behind each piece, and consider Pine Cone for gifts that celebrate culture, craft, and care. If you’re a marketing partner, investor, or potential team member, contact us to discuss collaborations, investments, or opportunities to join our growing craft ecosystem.